In [9]:
# SABX__dryrun.ipynb
# Цель: черновик безопасного обработчика задач PCReboot.
#
# Здесь мы НЕ делаем WinRM.
# Здесь мы принимаем RebootTask и возвращаем RebootResult.
# Пока только dry-run и проверка allowed_hosts.

Жизненный цикл (ЖЦ программы) при статус ПК = выкл:
1. Задача создана          → PENDING
2. Проверяем allowed_hosts → прошёл
3. Проверяем доступность   → ping не идёт, порт 5985 закрыт
4. Останавливаемся         → PRECHECK_FAILED

PENDING
  ↓
allowed_hosts? ──нет──→ NOT_ALLOWED
  ↓ да
dry_run? ──да──→ SKIPPED
  ↓ нет
pre-check (ping / порт) ──недоступен──→ PRECHECK_FAILED
  ↓ доступен
WinRM auth ──ошибка──→ AUTH_ERROR
  ↓ ок
отправка shutdown ──ошибка──→ COMMAND_ERROR
  ↓ отправлена
COMMAND_SENT
  ↓
наблюдение 60 сек ──пропал──→ REBOOT_CONFIRMED
  ↓ не пропал
TIMEOUT

1. Создать RebootResult со статусом PENDING.
2. Проверить, что host есть в ALLOWED_HOSTS.
3. Если нет — вернуть NOT_ALLOWED.
4. Если dry_run=True — вернуть SKIPPED с сообщением, что команда была бы отправлена.
5. Если dry_run=False — пока вернуть UNKNOWN_ERROR или COMMAND_ERROR с пометкой, что реальный backend ещё не подключён.

In [10]:
from models import RebootTask, RebootResult, RebootStatus
from datetime import datetime

# TODO: создать базовый RebootResult со статусом PENDING:

def process_task(task: RebootTask, allowed_hosts: set[str]) -> RebootResult: #WARN: task ТОЛЬКО ДЛЯ ВХОДНЫХ ЗАДАЧ от юзера, не состояние выполнения задачи
    """
    Принята task = RebootTask -> PROC:process_task()
    """
    # Пока только:
    # - проверка allowed_hosts;
    # - dry-run;
    # - без реального WinRM.

    res = RebootResult(
        host=task.host,
        run_id=task.run_id,
        status=RebootStatus.PENDING,
        message="Задача создана. Ожидание перезагрузки...",
        started_at=datetime.now(),
        dry_run=task.dry_run,
        method=task.method
    )
    
    # MADE: если task.host нет в allowed_hosts, вернуть NOT_ALLOWED
    if task.host not in allowed_hosts:
        res.status=RebootStatus.NOT_ALLOWED
        res.message = f"Host {task.host} isn't in allowed list"
        return res

    # MADE: если task.dry_run, вернуть SKIPPED
    # MADE: message должен показывать команду, которая была бы выполнена:
    # shutdown /r /t {task.reboot_delay_sec} /f
    if task.dry_run:
        res.status=RebootStatus.SKIPPED
        res.message=f"DRY-RUN: shutdown /r /t {task.reboot_delay_sec} /f"
        return res
        

    # MADE: если не dry-run, пока вернуть COMMAND_ERROR
    # message: "Real backend not implemented yet"    
    #STUB:
    res.status=RebootStatus.COMMAND_ERROR
    res.message="Real backend not implemented yet"
    return res

In [12]:
#TEST:
ALLOWED_HOSTS = (
    "WS-K534D",
    "WS-K534F",
)

task = RebootTask(
    host="WS-K534D",
    run_id="run0001",
    dry_run=True,
    timeout_sec=20
)
print(process_task(task=task, allowed_hosts=ALLOWED_HOSTS))

RebootResult(host='WS-K534D', run_id='run0001', status=<RebootStatus.SKIPPED: 'SKIPPED'>, message='DRY-RUN: shutdown /r /t 5 /f', started_at=datetime.datetime(2026, 8, 18, 16, 7, 57, 457887), finished_at=None, duration_ms=None, dry_run=True, method='winrm')


In [ ]:
#TEST: --- ТЕСТОВАЯ ПРОВЕРКА ---

ALLOWED_HOSTS = {"WS-K534D", "WS-K534F"}

tasks = [
    RebootTask(host="WS-K534D", run_id="run-0003", dry_run=True),
    RebootTask(host="BAD-PC", run_id="run-0003", dry_run=True),
    RebootTask(host="WS-K534F", run_id="run-0003", dry_run=False),
]

print("--- Результаты обработки задач ---")
for task in tasks:
    result = process_task(task, ALLOWED_HOSTS)
    # Выводим статус, хост и сообщение
    print(f"{result.status.value:20} | {result.host:15} | {result.message}")

--- Результаты обработки задач ---
SKIPPED              | WS-K534D        | DRY-RUN: shutdown /r /t 5 /f
NOT_ALLOWED          | BAD-PC          | Host BAD-PC isn't in allowed list
COMMAND_ERROR        | WS-K534F        | Real backend not implemented yet
